In [1]:
import os
import random
import sys
import cv2
import tqdm
import json
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
!pip install polars seaborn

import polars as pl

In [3]:
!nvidia-smi

Thu Apr 16 20:09:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               On  |   00000000:D6:00.0 Off |                  Off |
| 33%   59C    P2            150W /  230W |   11733MiB /  24564MiB |     67%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
os.chdir('/workspace')

In [5]:
df = pl.read_csv("vindr_mammogram/mammo_processed_merged1/mammo_processed_merged.csv")
df = df.to_pandas()
df["merged_image_path"] = (
    df["merged_image_path"]
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
)

In [6]:
df['cc_finding_categories'].value_counts().sort_index()

cc_finding_categories
['Architectural Distortion', 'Mass']                                                                   1
['Architectural Distortion']                                                                          40
['Asymmetry', 'Mass']                                                                                  1
['Asymmetry']                                                                                         20
['Focal Asymmetry']                                                                                  107
['Global Asymmetry']                                                                                  11
['Mass']                                                                                             443
['Nipple Retraction', 'Mass']                                                                          1
['Nipple Retraction', 'Skin Thickening', 'Mass']                                                       1
['Nipple Retraction']            

In [7]:
def get_combined_finding_6class(cc_findings, mlo_findings, cc_birads, mlo_birads):
    if isinstance(cc_findings, str):
        cc_findings = eval(cc_findings) if cc_findings else []
    if isinstance(mlo_findings, str):
        mlo_findings = eval(mlo_findings) if mlo_findings else []
    
    cc_findings = cc_findings or []
    mlo_findings = mlo_findings or []
    all_findings = set(cc_findings + mlo_findings)
    
    if len(all_findings) > 1 and 'No Finding' in all_findings:
        all_findings.remove('No Finding')
    
    high_suspicion_structural = {
        'Architectural Distortion',
        'Skin Thickening',
        'Skin Retraction',
        'Nipple Retraction'
    }
    
    asymmetry_findings = {
        'Focal Asymmetry',
        'Global Asymmetry',
        'Asymmetry'
    }
    
    has_mass = 'Mass' in all_findings
    has_calc = 'Suspicious Calcification' in all_findings
    has_structural = bool(all_findings & high_suspicion_structural)
    has_asymmetry = bool(all_findings & asymmetry_findings)
    has_lymph = 'Suspicious Lymph Node' in all_findings
    
    def parse_birads(birads_str):
        if pd.isna(birads_str) or birads_str == '':
            return 0
        if isinstance(birads_str, str):
            try:
                return int(birads_str.strip().split()[-1])
            except:
                return 0
        return int(birads_str)
    
    cc_birads_num = parse_birads(cc_birads)
    mlo_birads_num = parse_birads(mlo_birads)
    max_birads = max(cc_birads_num, mlo_birads_num)
    
    if not all_findings or all_findings == {'No Finding'}:
        if max_birads == 1:
            return 0
        elif max_birads == 2:
            return 1
        else:
            return 1 if max_birads == 3 else 4
    
    if (has_mass and has_calc) or has_lymph:
        return 4  # Complex/Combined
    
    # Priority 5: Architectural distortions (without mass)
    if has_structural:
        return 5
    
    # Priority 3: Mass (isolated)
    if has_mass:
        return 3
    
    # Priority 2: Calcification (isolated)
    if has_calc:
        return 2
    
    if has_asymmetry and len(all_findings) == 1:
        return -1
    
    if has_asymmetry and len(all_findings) > 1:
        return 5
    
    print(f"Warning: Unknown finding combination: {all_findings}, BIRADS: {max_birads}")
    return 5

df['finding'] = df.apply(
    lambda row: get_combined_finding_6class(
        row['cc_finding_categories'], 
        row['mlo_finding_categories'],
        row['cc_breast_birads'],
        row['mlo_breast_birads']
    ),
    axis=1
)
df.drop(df[df['finding'] == -1].index, inplace=True)
df['finding'].value_counts().sort_index()

finding
0    6703
1    2329
2     127
3     483
4      85
5      83
Name: count, dtype: int64

In [8]:
import ast
import pandas as pd
from collections import Counter

ASYMMETRY_FINDINGS  = frozenset({"Asymmetry", "Focal Asymmetry", "Global Asymmetry"})
STRUCTURAL_FINDINGS = frozenset({"Architectural Distortion", "Skin Thickening", "Skin Retraction", "Nipple Retraction"})
FINDING_TO_BINARY   = [0, 1, 1, 1, 1, 1]
NUM_PROTOTYPES      = 6

def _parse_findings(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return frozenset()
    if isinstance(raw, (list, tuple, set)):
        return frozenset(str(f).strip() for f in raw if str(f).strip())
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw:
            return frozenset()
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, (list, tuple, set)):
                return frozenset(str(f).strip() for f in parsed if str(f).strip())
        except (ValueError, SyntaxError):
            pass
        return frozenset({raw})
    return frozenset()

def _parse_birads(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return 0
    if isinstance(raw, (int, float)):
        return int(raw)
    s = str(raw).strip()
    for token in reversed(s.split()):
        digits = "".join(c for c in token if c.isdigit())
        if digits:
            return int(digits[0])
    return 0

def add_finding_columns(df,
                        cc_findings_col="cc_finding_categories",
                        mlo_findings_col="mlo_finding_categories",
                        cc_birads_col="cc_breast_birads",
                        mlo_birads_col="mlo_breast_birads"):
    rows, drop_idx, other_log = [], [], []

    for idx, row in df.iterrows():
        cc       = _parse_findings(row[cc_findings_col])
        mlo      = _parse_findings(row[mlo_findings_col])
        combined = cc | mlo

        if len(combined) > 1 and "No Finding" in combined:
            combined = combined - {"No Finding"}

        non_asym = combined - ASYMMETRY_FINDINGS
        if combined and not non_asym:
            drop_idx.append(idx)
            continue
        combined = non_asym

        max_birads  = max(_parse_birads(row[cc_birads_col]), _parse_birads(row[mlo_birads_col]))
        is_negative = not combined or combined == {"No Finding"}

        KNOWN = {"No Finding", "Mass", "Suspicious Calcification", "Suspicious Lymph Node"}
        remaining = combined - KNOWN - STRUCTURAL_FINDINGS

        if not is_negative and remaining:
            other_log.extend(list(remaining))

        rows.append({
            "idx":                idx,
            "has_neg_b1":         int(is_negative and max_birads <= 1),
            "has_neg_b2":         int(is_negative and max_birads > 1),
            "has_mass":           int("Mass" in combined),
            "has_calc":           int("Suspicious Calcification" in combined),
            "has_structural":     int(bool(combined & STRUCTURAL_FINDINGS)),
            "has_lymph": int(not is_negative and (
                                      "Suspicious Lymph Node" in combined or bool(remaining))),
        })

    print("=== Top findings landing in has_lymph_or_other ===")
    for finding, count in Counter(other_log).most_common(20):
        print(f"  {finding}: {count}")

    encoded = pd.DataFrame(rows).set_index("idx")
    df = df.drop(index=drop_idx).join(encoded).reset_index(drop=True)
    return df

df = add_finding_columns(df)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]
print("\n=== Finding counts ===")
print(df[FINDING_COLS].sum())


=== Top findings landing in has_lymph_or_other ===

=== Finding counts ===
has_neg_b1        6703
has_neg_b2        2329
has_mass           570
has_calc           207
has_structural      90
has_lymph           22
dtype: int64


In [9]:
inbreast_df = pd.read_csv("inbreast_data/INbreast_merged/merged_metadata.csv")
inbreast_df["merged_image_path"] = (
    inbreast_df["merged_image_path"]
    .str.replace("INbreast Release 1.0", "inbreast_data", regex=False)
    .str.replace("vindr_original_data", "inbreast_data", regex=False))
inbreast_df['birads'] = inbreast_df['birads'].replace({'4a': '4', '4b': '4', '4c': '4','6':'5'})
inbreast_df['label'] = (inbreast_df['birads'].astype(int) - 1).astype(int)
inbreast_df['density'] = 0
inbreast_df['finding'] = 0
inbreast_df['cc_breast_birads'] = None
inbreast_df['mlo_breast_birads'] = None
inbreast_df['cc_breast_density'] = None
inbreast_df['device_id'] = 0
for col in ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph", "has_other"]:
    inbreast_df[col] = 0
inbreast_df.head()

,patient_id,laterality,merged_image_path,cc_file_name,mlo_file_name,cc_image_path,mlo_image_path,birads,cc_roi_width,cc_roi_height,...,mlo_breast_birads,cc_breast_density,device_id,has_neg_b1,has_neg_b2,has_mass,has_calc,has_structural,has_lymph,has_other
0,024ee3569b2605dc,L,inbreast_data/INbreast_merged/024ee3569b2605dc...,20588020,20588072,INbreast Release 1.0/INbreast_processed/205880...,INbreast Release 1.0/INbreast_processed/205880...,2,1557,3231,...,None,None,0,0,0,0,0,0,0,0
1,024ee3569b2605dc,R,inbreast_data/INbreast_merged/024ee3569b2605dc...,20587994,20588046,INbreast Release 1.0/INbreast_processed/205879...,INbreast Release 1.0/INbreast_processed/205880...,5,1535,3128,...,None,None,0,0,0,0,0,0,0,0
2,069212ec65a94339,L,inbreast_data/INbreast_merged/069212ec65a94339...,50994787,50994733,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,1,1226,2580,...,None,None,0,0,0,0,0,0,0,0
3,069212ec65a94339,R,inbreast_data/INbreast_merged/069212ec65a94339...,50994706,50994760,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,1,1128,2566,...,None,None,0,0,0,0,0,0,0,0
4,0b7396cdccacca82,L,inbreast_data/INbreast_merged/0b7396cdccacca82...,22670832,22670878,INbreast Release 1.0/INbreast_processed/226708...,INbreast Release 1.0/INbreast_processed/226708...,2,1627,2983,...,None,None,0,0,0,0,0,0,0,0


In [10]:
inbreast_df['label'].value_counts()

label
1    98
0    30
4    28
3    20
2    11
Name: count, dtype: int64

In [11]:
def birads_to_label(birads_category):
    """Convert BI-RADS categories to numerical labels 0-4 (for 5 classes)"""
    birads_num = int(birads_category.replace(" ", "")[-1])
    return birads_num - 1
df['label'] = df['cc_breast_birads'].apply(birads_to_label)

In [12]:
df['label'].value_counts()

label
0    6703
1    2337
3     339
2     319
4     112
Name: count, dtype: int64

In [13]:
df['cc_breast_density'].value_counts()

cc_breast_density
DENSITY C    7486
DENSITY D    1335
DENSITY B     942
DENSITY A      47
Name: count, dtype: int64

In [14]:
data = df[df['split'] == 'training']
test_df = df[df['split'] == 'test']

In [15]:
from sklearn.model_selection import train_test_split


study_meta = (
    data
    .groupby('study_id')
    .agg({
        'cc_breast_birads': 'first',   # BI-RADS at study level
        'finding': 'first'             # finding already encoded as 0–4
    })
    .reset_index()
)


# -------------------------------------------------
study_meta['stratify_key'] = (
    study_meta['cc_breast_birads'].astype(str) + '_' +
    study_meta['finding'].astype(str)
)


train_studies, val_studies = train_test_split(
    study_meta['study_id'],
    test_size=0.1,
    stratify=study_meta['stratify_key'],
    random_state=423
)

train_df = data[data['study_id'].isin(train_studies)].copy()
val_df   = data[data['study_id'].isin(val_studies)].copy()


In [16]:
train_df.shape

(7057, 32)

In [17]:
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as transforms
import random
import torch


def get_transforms(img_size=(512, 512)):
    """Enhanced mammogram preprocessing with medical imaging considerations"""
    
    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        
        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=(0.1, 0.1),
                scale=(0.9, 1.1),
                shear=6
            )
        ], p=0.6),
        
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([
            transforms.RandomPerspective(distortion_scale=0.1, p=1.0)
        ], p=0.1),
        
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=5.0, sigma=3.0)
        ], p=0.2),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1)
            )
        ], p=0.4),
        transforms.Lambda(lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2)) 
                         if random.random() < 0.4 else x),
        

        
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
        ], p=0.2),
        
                # NOISE AUGMENTATIONS
        transforms.Lambda(lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.02)) 
                         if random.random() < 0.4 else x),
        
        transforms.Lambda(lambda x: add_speckle_noise(x, std=random.uniform(0.01, 0.03)) 
                         if random.random() < 0.2 else x),
        transforms.ToTensor(),
        
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return train_transform, val_transform

def add_gaussian_noise(image, mean=0, std=0.02):
    """Gaussian noise - electronic noise in imaging sensors"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(mean, std, img_array.shape)
        noisy_img = np.clip(img_array + noise, 0, 1)
        return Image.fromarray((noisy_img * 255).astype(np.uint8))
    return image


def adjust_gamma(image, gamma=1.0):
    """
    Gamma correction - handles tissue density variation
    Gamma < 1 = brighter, > 1 = darker
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        gamma_corrected = np.power(img_array, gamma)
        return Image.fromarray((gamma_corrected * 255).astype(np.uint8))
    return image

In [18]:
import numpy as np
import random
from PIL import Image
import torchvision.transforms as T

# ── Noise helpers (unchanged — domain-correct) ────────────────────────────────
def add_gaussian_noise(image, mean=0, std=0.02):
    """Electronic sensor noise — additive Gaussian"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        noisy = np.clip(arr + np.random.normal(mean, std, arr.shape), 0, 1)
        return Image.fromarray((noisy * 255).astype(np.uint8))
    return image

def add_speckle_noise(image, std=0.03):
    """Multiplicative speckle — physically correct for mammography"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, std, arr.shape)
        noisy = np.clip(arr + arr * noise, 0, 1)
        return Image.fromarray((noisy * 255).astype(np.uint8))
    return image

def adjust_gamma(image, gamma=1.0):
    """Gamma correction — tissue density variation simulation"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        corrected = np.power(arr, gamma)
        return Image.fromarray((corrected * 255).astype(np.uint8))
    return image

def random_90_rotation(image):
    """
    Only 90°/180°/270° steps — avoids interpolation artifact on
    microcalcifications (Shia et al. 2024, Hamidinekoo et al. 2017)
    """
    k = random.choice([0, 1, 2, 3])
    return image.rotate(k * 90)

def get_transforms(img_size=(512, 512)):

    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),

        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomPerspective(distortion_scale=0.1, p=0.2),
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=15.0, sigma=4.0)
        ], p=0.25),

        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=None,
                scale=(0.9, 1.1),
                shear=0,
            )
        ], p=0.5),

        # Contrast + brightness — tissue density varies across patients
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1),
            )
        ], p=0.5),

        # Gamma — handles exposure variation between machines
        transforms.Lambda(
            lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2))
            if random.random() < 0.4 else x
        ),

        # Gaussian noise — simulates sensor noise, keep it very mild
        transforms.Lambda(
            lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.02))
            if random.random() < 0.3 else x
        ),

        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    return train_transform, val_transform

In [19]:
import os
import torch
import random
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

Image.MAX_IMAGE_PIXELS = None
torch.manual_seed(2024)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph", "has_other"]


class CustomDataset(Dataset):
    def __init__(self, df, transformations=None, ref_transform=None):
        self.df            = df.reset_index(drop=True)
        self.transform     = transformations
        self.ref_transform = ref_transform
        self.cls_names     = {0: "birads_0", 1: "birads_1",2: "birads_2",3: "birads_3",4: "birads_4"}
        self.cls_counts    = df['label'].value_counts().to_dict()

    def __len__(self):
        return len(self.df)

    def get_cls_name(self, label):
        return self.cls_names[label]


    def __getitem__(self, idx):
        row = self.df.iloc[idx]
    
        birads_label = int(row['label'])   # 0..4
        qry_label = 0 if birads_label == 0 else 1
    
        qry_density = row['cc_breast_density']
        qry_finding = row['finding']
    
        qry_im = Image.open(row['merged_image_path']).convert("RGB")
    
        if self.transform:
            qry_im = self.transform(qry_im)
    
        finding_vec = torch.tensor(
            [row[col] for col in FINDING_COLS], dtype=torch.float32
        )
    
        return {
            "qry_im":      qry_im,
    
            "qry_gt":      torch.tensor(qry_label, dtype=torch.long),
    
            "birads":      torch.tensor(birads_label, dtype=torch.long),
    
            "finding":     qry_finding,
            "finding_vec": finding_vec,
    
            "has_neg_b1":     torch.tensor(row["has_neg_b1"],     dtype=torch.long),
            "has_neg_b2":     torch.tensor(row["has_neg_b2"],     dtype=torch.long),
            "has_mass":       torch.tensor(row["has_mass"],       dtype=torch.long),
            "has_calc":       torch.tensor(row["has_calc"],       dtype=torch.long),
            "has_structural": torch.tensor(row["has_structural"], dtype=torch.long),
            "has_lymph":      torch.tensor(row["has_lymph"],      dtype=torch.long),
        }


def get_dls(train_df, val_df, test_df, inbreast_df, bs, ns=6):
    train_trans, val_trans = get_transforms(img_size=(512, 512))

    tr_ds       = CustomDataset(train_df,    train_trans, val_trans)
    vl_ds       = CustomDataset(val_df,      val_trans,   val_trans)
    test_ds     = CustomDataset(test_df,     val_trans,   val_trans)
    inbreast_ds = CustomDataset(inbreast_df, val_trans,   val_trans)
    train_df['binary'] = (train_df['label'] != 0).astype(int)
    labels                       = train_df['binary'].values
    unique_classes, class_counts = np.unique(labels, return_counts=True)
    beta                         = 0.5
    class_weights                = (1.0 / class_counts) ** beta
    class_weights                = class_weights / class_weights.sum() * len(unique_classes)
    sample_weights               = class_weights[labels]

    print("Class counts:",           dict(zip(unique_classes, class_counts)))
    print("Smoothed class weights:", np.round(class_weights, 3))

    sampler = WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(sample_weights),
        replacement = True,
    )

    tr_dl = DataLoader(
        tr_ds, batch_size=bs, 
        shuffle=True,
        # sampler=sampler,
        drop_last=True, num_workers=4, pin_memory=True, persistent_workers=True,
    )
    val_dl = DataLoader(
        vl_ds, batch_size=bs, shuffle=False,
        drop_last=False, num_workers=8, pin_memory=True, persistent_workers=True,
    )
    ts_dl       = DataLoader(test_ds,     batch_size=1, shuffle=False, num_workers=ns)
    inbreast_dl = DataLoader(inbreast_ds, batch_size=1, shuffle=False, num_workers=ns)

    return tr_dl, val_dl, ts_dl, inbreast_dl, tr_ds.cls_names, tr_ds.cls_counts

In [20]:
tr_dl, val_dl, ts_dl, inbreast_dl, classes, cls_counts = get_dls(train_df,val_df, test_df, inbreast_df, bs =16)

Class counts: {np.int64(0): np.int64(4824), np.int64(1): np.int64(2233)}
Smoothed class weights: [0.81 1.19]


In [ ]:
import os
import gc
import copy
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from tqdm import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

# =============================================================================
# Config
# =============================================================================

FINDING_COLS = [
    "has_neg_b1",      # finding 0
    "has_neg_b2",      # finding 1
    "has_mass",        # finding 2
    "has_calc",        # finding 3
    "has_structural",  # finding 4
    "has_lymph",       # finding 5
]
NUM_FINDINGS = 6


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


# =============================================================================
# Attention Pooling
# =============================================================================

class AttentionPool2d(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        h = max(in_channels // reduction, 32)
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, h, kernel_size=1, bias=False),
            nn.BatchNorm2d(h),
            nn.GELU(),
            nn.Conv2d(h, 1, kernel_size=1, bias=True),
        )

    def forward(self, x):
        w = F.softmax(self.attn(x).flatten(2), dim=-1)
        return (x.flatten(2) * w).sum(-1)


# =============================================================================
# Backbone Model
# =============================================================================

class FindingAwareMoCoModel(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone,
        emb_dim=128,
        dropout=0.4,
        num_classes=2,
    ):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.backbone      = backbone

        if "efficientnet" in self.backbone_name:
            self.num_features = backbone.classifier[1].in_features
            backbone.classifier[1] = nn.Identity()
            self.is_cnn = True

        elif "convnext" in self.backbone_name:
            self.num_features = backbone.classifier[2].in_features
            backbone.classifier[2] = nn.Identity()
            self.is_cnn = True

        elif "resnet" in self.backbone_name:
            self.num_features = backbone.fc.in_features
            backbone.fc = nn.Identity()
            self.is_cnn = True

        elif "densenet" in self.backbone_name:
            self.num_features = backbone.classifier.in_features
            backbone.classifier = nn.Identity()
            self.is_cnn = True

        elif "swin" in self.backbone_name:
            self.num_features = backbone.head.in_features
            backbone.head = nn.Identity()
            self.is_cnn = False

        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        self.pool = AttentionPool2d(self.num_features) if self.is_cnn else None

        # cls_head reads backbone features directly
        self.cls_head = nn.Sequential(
            nn.Linear(self.num_features, 512),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

        # proj_head reads backbone features for contrastive
        self.proj_head = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.BatchNorm1d(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

        self._init_weights()

    def _init_weights(self):
        for m in list(self.cls_head.modules()) + list(self.proj_head.modules()):
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
                if hasattr(m, "weight") and m.weight is not None:
                    nn.init.ones_(m.weight)
                if hasattr(m, "bias") and m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _extract(self, x):
        f = self.backbone(x)
        if isinstance(f, tuple):
            f = f[0]

        if self.is_cnn:
            if f.ndim == 4:
                f = self.pool(f) if self.pool is not None else f.flatten(2).mean(-1)
            elif f.ndim == 3:
                f = f.mean(1)
        else:
            if f.ndim == 3:
                f = f.mean(1)
            elif f.ndim == 4:
                f = f.flatten(2).mean(-1)

        return f

    def forward(self, x, return_embedding=False):
        feat   = self._extract(x)
        logits = self.cls_head(feat)

        if return_embedding:
            emb = self.proj_head(feat)
            emb = F.normalize(emb, dim=1)
            return logits, emb

        return logits


# =============================================================================
# MoCo Wrapper
# =============================================================================

class FindingAwareMoCo(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone_fn,
        backbone_weights,
        emb_dim=128,
        dropout=0.4,
        num_classes=2,
        num_findings=NUM_FINDINGS,
        queue_size=32,
        m=0.999,
        temperature=0.07,
    ):
        super().__init__()

        self.num_findings = num_findings
        self.queue_size   = queue_size
        self.m            = m
        self.temperature  = temperature
        self.emb_dim      = emb_dim

        q_backbone = backbone_fn(weights=backbone_weights)
        self.query_encoder = FindingAwareMoCoModel(
            backbone_name=backbone_name,
            backbone=q_backbone,
            emb_dim=emb_dim,
            dropout=dropout,
            num_classes=num_classes,
        )

        k_backbone = backbone_fn(weights=backbone_weights)
        self.key_encoder = FindingAwareMoCoModel(
            backbone_name=backbone_name,
            backbone=k_backbone,
            emb_dim=emb_dim,
            dropout=dropout,
            num_classes=num_classes,
        )

        self.key_encoder.load_state_dict(self.query_encoder.state_dict())
        for p in self.key_encoder.parameters():
            p.requires_grad = False

        self.register_buffer(
            "queues",
            F.normalize(torch.randn(num_findings, queue_size, emb_dim), dim=-1)
        )
        self.register_buffer(
            "queue_labels",
            torch.full((num_findings, queue_size), -1, dtype=torch.long)
        )
        self.register_buffer(
            "queue_valid",
            torch.zeros(num_findings, queue_size, dtype=torch.bool)
        )
        self.register_buffer(
            "queue_ptr",
            torch.zeros(num_findings, dtype=torch.long)
        )

    @torch.no_grad()
    def momentum_update_key_encoder(self):
        for param_q, param_k in zip(
            self.query_encoder.parameters(),
            self.key_encoder.parameters()
        ):
            param_k.data = param_k.data * self.m + param_q.data * (1.0 - self.m)

    @torch.no_grad()
    def dequeue_and_enqueue(self, keys, finding_vec, labels):
        B = keys.size(0)
        for i in range(B):
            active = torch.where(finding_vec[i] > 0.5)[0]
            if len(active) == 0:
                continue
            k = keys[i]
            l = labels[i].long()
            for f in active:
                f   = int(f.item())
                ptr = int(self.queue_ptr[f].item())
                self.queues[f, ptr]       = k
                self.queue_labels[f, ptr] = l
                self.queue_valid[f, ptr]  = True
                self.queue_ptr[f] = (ptr + 1) % self.queue_size

    def forward_query(self, x, return_embedding=False):
        return self.query_encoder(x, return_embedding=return_embedding)

    @torch.no_grad()
    def forward_key(self, x):
        _, k = self.key_encoder(x, return_embedding=True)
        return k


# =============================================================================
# Finding-aware Supervised Contrastive Loss — Binary
#
#   Same class  + Same finding  → strong positive   w=1.0
#   Same class  + Diff finding  → moderate positive  w=0.4
#   Diff class  (any finding)   → negative           w=0.0
#
#   Per-sample loss weight based on active findings:
#   finding 0,1 (neg_b1, neg_b2) → majority → low weight     (0.5)
#   finding 2,3 (mass, calc)     → minority → higher weight   (2.0)
#   finding 4,5 (structural,lymph)→ minority → highest weight (2.5)
#   sample_weight = max finding weight among active findings
# =============================================================================

class FindingAwareSupConLoss(nn.Module):
    def __init__(
        self,
        temperature=0.07,
        same_class_same_finding=1.0,
        same_class_diff_finding=0.4,
        finding_loss_weights=(0.5, 0.5, 2.0, 2.0, 2.5, 2.5),
    ):
        super().__init__()
        self.tau                  = temperature
        self.w_sc_sf              = same_class_same_finding
        self.w_sc_df              = same_class_diff_finding
        self.finding_loss_weights = finding_loss_weights

    def forward(self, q, queues, queue_labels, queue_valid, finding_vec, labels):
        """
        q:             [B, D]
        queues:        [F, K, D]
        queue_labels:  [F, K]
        queue_valid:   [F, K]
        finding_vec:   [B, F]
        labels:        [B]
        """
        B, D     = q.shape
        F_, K, _ = queues.shape
        losses       = []
        loss_weights = []

        for i in range(B):
            qi          = q[i]
            li          = labels[i].long()
            active_list = torch.where(finding_vec[i] > 0.5)[0].tolist()

            if len(active_list) == 0:
                continue

            # sample weight = max finding weight among active findings
            # pathological findings dominate over majority findings
            sample_weight = max(
                self.finding_loss_weights[f] for f in active_list
            )

            all_logits   = []
            all_weights  = []
            has_positive = False

            for f in range(F_):
                for k in range(K):
                    if not queue_valid[f, k]:
                        continue

                    sim          = torch.dot(qi, queues[f, k]) / self.tau
                    same_class   = (li == queue_labels[f, k].long())
                    same_finding = (f in active_list)

                    if same_class and same_finding:
                        w = self.w_sc_sf       # strong positive
                        has_positive = True
                    elif same_class and not same_finding:
                        w = self.w_sc_df       # moderate positive
                        has_positive = True
                    else:
                        w = 0.0                # negative — denominator only

                    all_logits.append(sim)
                    all_weights.append(w)

            if not has_positive or len(all_logits) == 0:
                continue

            all_logits  = torch.stack(all_logits)
            all_weights = torch.tensor(
                all_weights, device=q.device, dtype=q.dtype
            )

            log_denom   = torch.logsumexp(all_logits, dim=0)
            pos_mask    = all_weights > 0
            pos_logits  = all_logits[pos_mask]
            pos_weights = all_weights[pos_mask]

            loss_i = -(pos_weights * (pos_logits - log_denom)).sum() / (
                pos_weights.sum() + 1e-8
            )

            losses.append(loss_i)
            loss_weights.append(sample_weight)

        if len(losses) == 0:
            return torch.tensor(0.0, device=q.device)

        losses       = torch.stack(losses)
        loss_weights = torch.tensor(
            loss_weights, device=q.device, dtype=q.dtype
        )

        # weighted mean — pathological finding samples contribute more
        return (loss_weights * losses).sum() / (loss_weights.sum() + 1e-8)


# =============================================================================
# Classification Loss
# =============================================================================

class AsymmetricFocalLoss(nn.Module):
    def __init__(self, gamma_pos=1.0, gamma_neg=2.0, alpha=0.6):
        super().__init__()
        self.gp    = gamma_pos
        self.gn    = gamma_neg
        self.alpha = alpha

    def forward(self, logits, targets):
        targets = targets.long()
        ce      = F.cross_entropy(logits, targets, reduction="none")
        pt      = torch.exp(-ce)

        gamma = torch.where(
            targets == 1,
            torch.full_like(ce, self.gp),
            torch.full_like(ce, self.gn),
        )
        alpha_t = torch.where(
            targets == 1,
            torch.full_like(ce, self.alpha),
            torch.full_like(ce, 1.0 - self.alpha),
        )
        return (alpha_t * ((1.0 - pt) ** gamma) * ce).mean()


# =============================================================================
# Validation
# =============================================================================

@torch.no_grad()
def validate(model, loader, device, cls_loss_fn, class_names=None):
    class_names = class_names or ["Benign", "Malignant"]
    model.eval()

    total_loss = 0.0
    preds, targets = [], []

    for batch in loader:
        imgs   = batch["qry_im"].to(device, non_blocking=True)
        labels = batch["qry_gt"].to(device, non_blocking=True).long()

        logits = model.forward_query(imgs, return_embedding=False)
        total_loss += cls_loss_fn(logits, labels).item()

        pred = logits.argmax(1)
        preds.extend(pred.cpu().numpy())
        targets.extend(labels.cpu().numpy())

    val_f1 = f1_score(targets, preds, average="binary", pos_label=1, zero_division=0)

    print("\n--- Validation ---")
    print(classification_report(targets, preds, target_names=class_names, zero_division=0))

    return total_loss / max(len(loader), 1), val_f1


# =============================================================================
# Test
# =============================================================================

@torch.no_grad()
def test_model(model, loader, device, save_dir, name="Test", class_names=None):
    class_names = class_names or ["Benign", "Malignant"]
    model.eval()

    preds, labels = [], []

    for batch in loader:
        imgs = batch["qry_im"].to(device, non_blocking=True)
        gt   = batch["qry_gt"].to(device, non_blocking=True).long()

        logits = model.forward_query(imgs, return_embedding=False)
        pred   = logits.argmax(1)

        preds.extend(pred.cpu().numpy())
        labels.extend(gt.cpu().numpy())

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="binary", pos_label=1, zero_division=0)
    cm  = confusion_matrix(labels, preds)

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, 0

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"\n{'='*70}")
    print(f"{name} | Acc={acc:.4f}  F1={f1:.4f}  Sens={sens:.4f}  Spec={spec:.4f}")
    print(cm)
    print(classification_report(labels, preds, target_names=class_names, zero_division=0))
    print('=' * 70)

    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, f"{name.lower().replace(' ', '_')}_report.txt"), "w") as fh:
        fh.write(f"{name}\n")
        fh.write(f"Acc={acc:.4f}  F1={f1:.4f}  Sens={sens:.4f}  Spec={spec:.4f}\n\n")
        fh.write(str(cm) + "\n\n")
        fh.write(classification_report(labels, preds, target_names=class_names, zero_division=0))

    return acc, f1


# =============================================================================
# Training Loop
# =============================================================================

def train_model(
    model,
    train_loader,
    val_loader,
    device,
    lr_backbone=1e-4,
    lr_heads=1e-4,
    epochs=60,
    patience=15,
    save_path="best_model.pth",
    con_weight=0.3,
    warmup_epochs=3,
    ramp_epochs=6,
    class_names=None,
):
    class_names = class_names or ["Benign", "Malignant"]

    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    log_path = save_path.replace(".pth", "_log.txt")

    cls_loss_fn = AsymmetricFocalLoss(
        gamma_pos=1.0,
        gamma_neg=2.0,
        alpha=0.6,
    ).to(device)

    con_loss_fn = FindingAwareSupConLoss(
        temperature=model.temperature,
        same_class_same_finding=1.0,
        same_class_diff_finding=0.5,
        finding_loss_weights=(0.05, 0.1, 0.7, 0.7, 0.8, 0.8),
    ).to(device)

    optimizer = AdamW([
        {
            "params": model.query_encoder.backbone.parameters(),
            "lr": lr_backbone,
            "weight_decay": 0.05,
        },
        {
            "params": model.query_encoder.cls_head.parameters(),
            "lr": lr_heads,
            "weight_decay": 0.05,
        },
        {
            "params": model.query_encoder.proj_head.parameters(),
            "lr": lr_heads,
            "weight_decay": 0.05,
        },
    ])

    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    best_f1      = 0.0
    not_improved = 0

    for epoch in range(epochs):
        if epoch < warmup_epochs:
            cw = 0.0
        else:
            cw = con_weight * min(
                1.0,
                (epoch - warmup_epochs + 1) / max(ramp_epochs, 1)
            )

        model.train()
        e_cls = 0.0
        e_con = 0.0

        all_preds  = []
        all_labels = []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch in pbar:
            imgs          = batch["qry_im"].to(device, non_blocking=True)
            binary_labels = batch["qry_gt"].to(device, non_blocking=True).long()
            finding_vec   = batch["finding_vec"].to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=(scaler is not None)):
                logits, q = model.forward_query(imgs, return_embedding=True)

                cls_loss = cls_loss_fn(logits, binary_labels)

                if cw > 0:
                    con_loss = con_loss_fn(
                        q=q,
                        queues=model.queues,
                        queue_labels=model.queue_labels,
                        queue_valid=model.queue_valid,
                        finding_vec=finding_vec,
                        labels=binary_labels,
                    )
                else:
                    con_loss = torch.tensor(0.0, device=device)

                total_loss = cls_loss + cw * con_loss

            if scaler is not None:
                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.query_encoder.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.query_encoder.parameters(), 1.0)
                optimizer.step()

            with torch.no_grad():
                model.momentum_update_key_encoder()
                k = model.forward_key(imgs)
                model.dequeue_and_enqueue(k, finding_vec, binary_labels)

            pred = logits.argmax(1)
            all_preds.extend(pred.detach().cpu().numpy())
            all_labels.extend(binary_labels.detach().cpu().numpy())

            e_cls += cls_loss.item()
            e_con += con_loss.item()

            pbar.set_postfix(
                cls=f"{cls_loss.item():.3f}",
                con=f"{con_loss.item():.3f}",
                cw=f"{cw:.2f}",
            )

        n = max(len(train_loader), 1)

        print(f"\n{'='*70}")
        print(f"Epoch {epoch+1}/{epochs}  cls={e_cls/n:.4f}  con={e_con/n:.4f}  cw={cw:.4f}")
        print("\n--- Train ---")
        print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

        val_loss, val_f1 = validate(
            model=model,
            loader=val_loader,
            device=device,
            cls_loss_fn=cls_loss_fn,
            class_names=class_names,
        )

        print(f"Val loss={val_loss:.4f}  Val F1={val_f1:.4f}")
        print('=' * 70)

        scheduler.step()

        with open(log_path, "a") as fh:
            fh.write(
                f"Epoch {epoch+1}  "
                f"cls={e_cls/n:.4f}  con={e_con/n:.4f}  "
                f"cw={cw:.4f}  val_f1={val_f1:.4f}\n"
            )

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                "epoch":            epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state":  optimizer.state_dict(),
                "best_f1":          best_f1,
            }, save_path)
            print(f"Saved best model with F1={best_f1:.4f}")
            not_improved = 0
        else:
            not_improved += 1
            if not_improved >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    return best_f1


# =============================================================================
# Runner
# =============================================================================

def run_experiments(train_loader, val_loader, test_loader, inbreast_loader):
    seed_everything(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    config = dict(
        lr_backbone=1e-4,
        lr_heads=1e-4,
        epochs=60,
        patience=15,
        con_weight=0.3,
        warmup_epochs=3,
        ramp_epochs=6,
        class_names=["Benign", "Malignant"],
    )

    backbones = [
        {
            "name": "efficientnet_b3",
            "fn": models.efficientnet_b3,
            "w": models.EfficientNet_B3_Weights.DEFAULT,
        },
        {
            "name": "convnext_base",
            "fn": models.convnext_base,
            "w": models.ConvNeXt_Base_Weights.DEFAULT,
        },
    ]

    all_results = {}

    for cfg in backbones:
        out_dir   = f"/workspace/Thesis_updated_results/binary_512_finding_moco/{cfg['name']}"
        save_path = f"{out_dir}/best_model.pth"
        os.makedirs(out_dir, exist_ok=True)

        print(f"\n{'#'*70}")
        print(cfg["name"])
        print(f"{'#'*70}")

        model = FindingAwareMoCo(
            backbone_name=cfg["name"],
            backbone_fn=cfg["fn"],
            backbone_weights=cfg["w"],
            emb_dim=128,
            dropout=0.4,
            num_classes=2,
            num_findings=NUM_FINDINGS,
            queue_size=32,
            m=0.999,
            temperature=0.07,
        ).to(device)

        num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable Params: {num_params:,}")

        best_f1 = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            save_path=save_path,
            **config,
        )

        ckpt = torch.load(save_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"Loaded epoch {ckpt['epoch']+1} | best F1={ckpt['best_f1']:.4f}")

        acc_v, f1_v = test_model(
            model=model,
            loader=test_loader,
            device=device,
            save_dir=out_dir,
            name="VinDr-Test",
            class_names=config["class_names"],
        )

        acc_i, f1_i = test_model(
            model=model,
            loader=inbreast_loader,
            device=device,
            save_dir=out_dir,
            name="INbreast",
            class_names=config["class_names"],
        )

        all_results[cfg["name"]] = dict(
            best_val_f1=best_f1,
            vindr_acc=acc_v,
            vindr_f1=f1_v,
            inbreast_acc=acc_i,
            inbreast_f1=f1_i,
        )

        del model, ckpt
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n{'='*70}")
    print("FINAL RESULTS")
    print(f"{'='*70}")
    print(f"{'Model':<22} {'ValF1':>8} {'VinDr-F1':>10} {'INbreast-F1':>13}")
    print("-" * 60)

    for name, r in all_results.items():
        print(
            f"{name:<22} "
            f"{r['best_val_f1']:>8.4f} "
            f"{r['vindr_f1']:>10.4f} "
            f"{r['inbreast_f1']:>13.4f}"
        )

    return all_results


results = run_experiments(tr_dl, val_dl, ts_dl, inbreast_dl)

Device: cuda

######################################################################
efficientnet_b3
######################################################################
Trainable Params: 14,636,843


Epoch 1/60:  34%|███▍      | 150/441 [03:56<04:10,  1.16it/s, cls=0.126, con=0.000, cw=0.00]  

In [ ]:
ddddddddddddddd

In [ ]:
import os
import gc
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from tqdm import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

# =============================================================================
# Config
# =============================================================================

FINDING_COLS = [
    "has_neg_b1",      # finding 0
    "has_neg_b2",      # finding 1
    "has_mass",        # finding 2
    "has_calc",        # finding 3
    "has_structural",  # finding 4
    "has_lymph",       # finding 5
]
NUM_FINDINGS = 6
NUM_CLASSES = 5
BIRADS_CLASS_NAMES = ["BI-RADS 1", "BI-RADS 2", "BI-RADS 3", "BI-RADS 4", "BI-RADS 5"]


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


# =============================================================================
# Attention Pooling
# =============================================================================

class AttentionPool2d(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        h = max(in_channels // reduction, 32)
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, h, kernel_size=1, bias=False),
            nn.BatchNorm2d(h),
            nn.GELU(),
            nn.Conv2d(h, 1, kernel_size=1, bias=True),
        )

    def forward(self, x):
        # x: [B, C, H, W]
        w = F.softmax(self.attn(x).flatten(2), dim=-1)   # [B,1,HW]
        return (x.flatten(2) * w).sum(-1)                # [B,C]


# =============================================================================
# Backbone Model
#   - 5-class BI-RADS classifier
#   - finding-aware MoCo only in embedding branch
# =============================================================================

class FindingAwareMoCoModel(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone,
        emb_dim=128,
        dropout=0.4,
        num_classes=5,
    ):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.backbone = backbone

        if "efficientnet" in self.backbone_name:
            self.num_features = backbone.classifier[1].in_features
            backbone.classifier[1] = nn.Identity()
            self.is_cnn = True

        elif "convnext" in self.backbone_name:
            self.num_features = backbone.classifier[2].in_features
            backbone.classifier[2] = nn.Identity()
            self.is_cnn = True

        elif "resnet" in self.backbone_name:
            self.num_features = backbone.fc.in_features
            backbone.fc = nn.Identity()
            self.is_cnn = True

        elif "densenet" in self.backbone_name:
            self.num_features = backbone.classifier.in_features
            backbone.classifier = nn.Identity()
            self.is_cnn = True

        elif "swin" in self.backbone_name:
            self.num_features = backbone.head.in_features
            backbone.head = nn.Identity()
            self.is_cnn = False

        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        self.pool = AttentionPool2d(self.num_features) if self.is_cnn else None

        self.cls_head = nn.Sequential(
            nn.Linear(self.num_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

        self.proj_head = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.BatchNorm1d(self.num_features),
            nn.GELU(),
            nn.Linear(self.num_features, emb_dim),
        )

        self._init_weights()

    def _init_weights(self):
        for m in list(self.cls_head.modules()) + list(self.proj_head.modules()):
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
                if hasattr(m, "weight") and m.weight is not None:
                    nn.init.ones_(m.weight)
                if hasattr(m, "bias") and m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _extract(self, x):
        f = self.backbone(x)
        if isinstance(f, tuple):
            f = f[0]

        if self.is_cnn:
            if f.ndim == 4:
                f = self.pool(f) if self.pool is not None else f.flatten(2).mean(-1)
            elif f.ndim == 3:
                f = f.mean(1)
        else:
            if f.ndim == 3:
                f = f.mean(1)
            elif f.ndim == 4:
                f = f.flatten(2).mean(-1)

        return f

    def forward(self, x, return_embedding=False):
        feat = self._extract(x)           # [B, D]
        logits = self.cls_head(feat)      # [B, 5]

        if return_embedding:
            emb = self.proj_head(feat)
            emb = F.normalize(emb, dim=1)
            return logits, emb

        return logits


# =============================================================================
# Finding-aware MoCo Wrapper
#   - Queue remains per finding
#   - Classifier is now 5-class BI-RADS
# =============================================================================

class FindingAwareMoCo(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone_fn,
        backbone_weights,
        emb_dim=128,
        dropout=0.4,
        num_classes=5,
        num_findings=NUM_FINDINGS,
        queue_size=32,
        m=0.999,
        temperature=0.07,
    ):
        super().__init__()

        self.num_findings = num_findings
        self.queue_size = queue_size
        self.m = m
        self.temperature = temperature
        self.emb_dim = emb_dim

        q_backbone = backbone_fn(weights=backbone_weights)
        self.query_encoder = FindingAwareMoCoModel(
            backbone_name=backbone_name,
            backbone=q_backbone,
            emb_dim=emb_dim,
            dropout=dropout,
            num_classes=num_classes,
        )

        k_backbone = backbone_fn(weights=backbone_weights)
        self.key_encoder = FindingAwareMoCoModel(
            backbone_name=backbone_name,
            backbone=k_backbone,
            emb_dim=emb_dim,
            dropout=dropout,
            num_classes=num_classes,
        )

        self.key_encoder.load_state_dict(self.query_encoder.state_dict())

        for p in self.key_encoder.parameters():
            p.requires_grad = False

        self.register_buffer(
            "queues",
            F.normalize(torch.randn(num_findings, queue_size, emb_dim), dim=-1)
        )
        self.register_buffer(
            "queue_ptr",
            torch.zeros(num_findings, dtype=torch.long)
        )

    @torch.no_grad()
    def momentum_update_key_encoder(self):
        for param_q, param_k in zip(
            self.query_encoder.parameters(),
            self.key_encoder.parameters()
        ):
            param_k.data = param_k.data * self.m + param_q.data * (1.0 - self.m)

    @torch.no_grad()
    def dequeue_and_enqueue(self, keys, finding_vec):
        """
        keys: [B, D]
        finding_vec: [B, F] multi-hot
        """
        B = keys.size(0)

        for i in range(B):
            active = torch.where(finding_vec[i] > 0.5)[0]
            if len(active) == 0:
                continue

            k = keys[i]

            for f in active:
                f = int(f.item())
                ptr = int(self.queue_ptr[f].item())
                self.queues[f, ptr] = k
                self.queue_ptr[f] = (ptr + 1) % self.queue_size

    def forward_query(self, x, return_embedding=False):
        return self.query_encoder(x, return_embedding=return_embedding)

    @torch.no_grad()
    def forward_key(self, x):
        _, k = self.key_encoder(x, return_embedding=True)
        return k


# =============================================================================
# Finding-aware MoCo Loss
#   positives = active finding queues
#   negatives = all inactive finding queues
# =============================================================================

class FindingAwareMoCoLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.tau = temperature

    def forward(self, q, queues, finding_vec):
        """
        q: [B, D]
        queues: [F, K, D]
        finding_vec: [B, F]
        """
        B, D = q.shape
        F_, K, D_ = queues.shape
        assert D == D_

        losses = []

        for i in range(B):
            qi = q[i]
            active_findings = torch.where(finding_vec[i] > 0.5)[0]

            if len(active_findings) == 0:
                continue

            pos_bank = queues[active_findings].reshape(-1, D)

            inactive_findings = torch.tensor(
                [f for f in range(F_) if f not in active_findings.tolist()],
                device=q.device,
                dtype=torch.long,
            )

            neg_bank = None
            if len(inactive_findings) > 0:
                neg_bank = queues[inactive_findings].reshape(-1, D)

            pos_logits = torch.matmul(pos_bank, qi) / self.tau

            if neg_bank is not None and neg_bank.numel() > 0:
                neg_logits = torch.matmul(neg_bank, qi) / self.tau
                logits = torch.cat([pos_logits, neg_logits], dim=0)
            else:
                logits = pos_logits

            log_denom = torch.logsumexp(logits, dim=0)
            loss_i = -(pos_logits - log_denom).mean()
            losses.append(loss_i)

        if len(losses) == 0:
            return torch.tensor(0.0, device=q.device)

        return torch.stack(losses).mean()


# =============================================================================
# Multiclass Loss
#   You can pass class_weights if you have imbalance
# =============================================================================

class WeightedCrossEntropyLoss(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        if class_weights is not None:
            class_weights = torch.tensor(class_weights, dtype=torch.float)
            self.register_buffer("class_weights", class_weights)
        else:
            self.class_weights = None

    def forward(self, logits, targets):
        targets = targets.long()
        return F.cross_entropy(logits, targets, weight=self.class_weights)


# =============================================================================
# Validation
# =============================================================================

@torch.no_grad()
def validate(model, loader, device, cls_loss_fn, class_names=None):
    class_names = class_names or BIRADS_CLASS_NAMES
    model.eval()

    total_loss = 0.0
    preds, targets = [], []

    for batch in loader:
        imgs = batch["qry_im"].to(device, non_blocking=True)
        labels = batch["qry_gt"].to(device, non_blocking=True).long()

        logits = model.forward_query(imgs, return_embedding=False)
        total_loss += cls_loss_fn(logits, labels).item()

        pred = logits.argmax(1)
        preds.extend(pred.cpu().numpy())
        targets.extend(labels.cpu().numpy())

    acc = accuracy_score(targets, preds)
    macro_f1 = f1_score(targets, preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(targets, preds, average="weighted", zero_division=0)

    print("\n--- Validation ---")
    print(classification_report(targets, preds, target_names=class_names, zero_division=0))

    return {
        "loss": total_loss / max(len(loader), 1),
        "acc": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }


# =============================================================================
# Test
# =============================================================================

@torch.no_grad()
def test_model(model, loader, device, save_dir, name="Test", class_names=None):
    class_names = class_names or BIRADS_CLASS_NAMES
    model.eval()

    preds, labels = [], []

    for batch in loader:
        imgs = batch["qry_im"].to(device, non_blocking=True)
        gt = batch["qry_gt"].to(device, non_blocking=True).long()

        logits = model.forward_query(imgs, return_embedding=False)
        pred = logits.argmax(1)

        preds.extend(pred.cpu().numpy())
        labels.extend(gt.cpu().numpy())

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)
    cm = confusion_matrix(labels, preds)

    print(f"\n{'='*70}")
    print(f"{name} | Acc={acc:.4f}  Macro-F1={macro_f1:.4f}  Weighted-F1={weighted_f1:.4f}")
    print(cm)
    print(classification_report(labels, preds, target_names=class_names, zero_division=0))
    print('=' * 70)

    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, f"{name.lower().replace(' ', '_')}_report.txt"), "w") as fh:
        fh.write(f"{name}\n")
        fh.write(f"Acc={acc:.4f}  Macro-F1={macro_f1:.4f}  Weighted-F1={weighted_f1:.4f}\n\n")
        fh.write(str(cm) + "\n\n")
        fh.write(classification_report(labels, preds, target_names=class_names, zero_division=0))

    return {
        "acc": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }


# =============================================================================
# Training Loop
# =============================================================================

def train_model(
    model,
    train_loader,
    val_loader,
    device,
    lr_backbone=1e-4,
    lr_heads=1e-4,
    epochs=60,
    patience=15,
    save_path="best_model.pth",
    con_weight=0.30,
    warmup_epochs=3,
    ramp_epochs=3,
    class_names=None,
    class_weights=None,
):
    class_names = class_names or BIRADS_CLASS_NAMES

    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    log_path = save_path.replace(".pth", "_log.txt")

    cls_loss_fn = WeightedCrossEntropyLoss(class_weights=class_weights).to(device)
    con_loss_fn = FindingAwareMoCoLoss(temperature=model.temperature).to(device)

    optimizer = AdamW([
        {
            "params": model.query_encoder.backbone.parameters(),
            "lr": lr_backbone,
            "weight_decay": 0.05,
        },
        {
            "params": model.query_encoder.cls_head.parameters(),
            "lr": lr_heads,
            "weight_decay": 0.05,
        },
        {
            "params": model.query_encoder.proj_head.parameters(),
            "lr": lr_heads,
            "weight_decay": 0.05,
        },
    ])

    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    best_macro_f1 = 0.0
    not_improved = 0

    for epoch in range(epochs):
        if epoch < warmup_epochs:
            cw = 0.0
        else:
            cw = con_weight * min(
                1.0,
                (epoch - warmup_epochs + 1) / max(ramp_epochs, 1)
            )

        model.train()
        e_cls = 0.0
        e_con = 0.0

        all_preds = []
        all_labels = []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch in pbar:
            imgs = batch["qry_im"].to(device, non_blocking=True)
            labels = batch["qry_gt"].to(device, non_blocking=True).long()
            finding_vec = batch["finding_vec"].to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=(scaler is not None)):
                logits, q = model.forward_query(imgs, return_embedding=True)

                cls_loss = cls_loss_fn(logits, labels)

                if cw > 0:
                    con_loss = con_loss_fn(q, model.queues, finding_vec)
                else:
                    con_loss = torch.tensor(0.0, device=device)

                total_loss = cls_loss + cw * con_loss

            if scaler is not None:
                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.query_encoder.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.query_encoder.parameters(), 1.0)
                optimizer.step()

            with torch.no_grad():
                model.momentum_update_key_encoder()
                k = model.forward_key(imgs)
                model.dequeue_and_enqueue(k, finding_vec)

            pred = logits.argmax(1)
            all_preds.extend(pred.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

            e_cls += cls_loss.item()
            e_con += con_loss.item()

            pbar.set_postfix(
                cls=f"{cls_loss.item():.3f}",
                con=f"{con_loss.item():.3f}",
                cw=f"{cw:.2f}",
            )

        n = max(len(train_loader), 1)
        train_acc = accuracy_score(all_labels, all_preds)
        train_macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        train_weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

        print(f"\n{'='*70}")
        print(
            f"Epoch {epoch+1}/{epochs}  "
            f"cls={e_cls/n:.4f}  con={e_con/n:.4f}  cw={cw:.4f}  "
            f"train_acc={train_acc:.4f}  train_macro_f1={train_macro_f1:.4f}"
        )
        print("\n--- Train ---")
        print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

        val_metrics = validate(
            model=model,
            loader=val_loader,
            device=device,
            cls_loss_fn=cls_loss_fn,
            class_names=class_names,
        )

        print(
            f"Val loss={val_metrics['loss']:.4f}  "
            f"Val acc={val_metrics['acc']:.4f}  "
            f"Val macro_f1={val_metrics['macro_f1']:.4f}  "
            f"Val weighted_f1={val_metrics['weighted_f1']:.4f}"
        )
        print('=' * 70)

        scheduler.step(epoch + 1)

        with open(log_path, "a") as fh:
            fh.write(
                f"Epoch {epoch+1}  "
                f"cls={e_cls/n:.4f}  con={e_con/n:.4f}  cw={cw:.4f}  "
                f"train_acc={train_acc:.4f}  train_macro_f1={train_macro_f1:.4f}  "
                f"train_weighted_f1={train_weighted_f1:.4f}  "
                f"val_loss={val_metrics['loss']:.4f}  "
                f"val_acc={val_metrics['acc']:.4f}  "
                f"val_macro_f1={val_metrics['macro_f1']:.4f}  "
                f"val_weighted_f1={val_metrics['weighted_f1']:.4f}\n"
            )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_macro_f1": best_macro_f1,
            }, save_path)
            print(f"Saved best model with macro-F1={best_macro_f1:.4f}")
            not_improved = 0
        else:
            not_improved += 1
            if not_improved >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    return best_macro_f1


# =============================================================================
# Runner
# =============================================================================

def run_experiments(
    train_loader,
    val_loader,
    test_loader,
    inbreast_loader,
    class_weights=None,
):
    seed_everything(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    config = dict(
        lr_backbone=1e-4,
        lr_heads=1e-4,
        epochs=60,
        patience=15,
        con_weight=0.30,
        warmup_epochs=3,
        ramp_epochs=3,
        class_names=BIRADS_CLASS_NAMES,
        class_weights=class_weights,
    )

    backbones = [
        {
            "name": "efficientnet_b3",
            "fn": models.efficientnet_b3,
            "w": models.EfficientNet_B3_Weights.DEFAULT,
        },
        {
            "name": "convnext_base",
            "fn": models.convnext_base,
            "w": models.ConvNeXt_Base_Weights.DEFAULT,
        },
    ]

    all_results = {}

    for cfg in backbones:
        out_dir = f"/workspace/Thesis_updated_results/birads_512_finding_moco/{cfg['name']}"
        save_path = f"{out_dir}/best_model.pth"
        os.makedirs(out_dir, exist_ok=True)

        print(f"\n{'#'*70}")
        print(cfg["name"])
        print(f"{'#'*70}")

        model = FindingAwareMoCo(
            backbone_name=cfg["name"],
            backbone_fn=cfg["fn"],
            backbone_weights=cfg["w"],
            emb_dim=128,
            dropout=0.4,
            num_classes=NUM_CLASSES,
            num_findings=NUM_FINDINGS,
            queue_size=32,
            m=0.999,
            temperature=0.07,
        ).to(device)

        num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable Params: {num_params:,}")

        best_macro_f1 = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            save_path=save_path,
            **config,
        )

        ckpt = torch.load(save_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"Loaded epoch {ckpt['epoch']+1} | best macro-F1={ckpt['best_macro_f1']:.4f}")

        vindr_metrics = test_model(
            model=model,
            loader=test_loader,
            device=device,
            save_dir=out_dir,
            name="VinDr-Test",
            class_names=BIRADS_CLASS_NAMES,
        )

        inbreast_metrics = test_model(
            model=model,
            loader=inbreast_loader,
            device=device,
            save_dir=out_dir,
            name="INbreast",
            class_names=BIRADS_CLASS_NAMES,
        )

        all_results[cfg["name"]] = dict(
            best_val_macro_f1=best_macro_f1,
            vindr_acc=vindr_metrics["acc"],
            vindr_macro_f1=vindr_metrics["macro_f1"],
            vindr_weighted_f1=vindr_metrics["weighted_f1"],
            inbreast_acc=inbreast_metrics["acc"],
            inbreast_macro_f1=inbreast_metrics["macro_f1"],
            inbreast_weighted_f1=inbreast_metrics["weighted_f1"],
        )

        del model, ckpt
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n{'='*85}")
    print("FINAL RESULTS")
    print(f"{'='*85}")
    print(
        f"{'Model':<22} "
        f"{'Val Macro-F1':>12} "
        f"{'VinDr Macro-F1':>15} "
        f"{'INbreast Macro-F1':>18}"
    )
    print("-" * 85)

    for name, r in all_results.items():
        print(
            f"{name:<22} "
            f"{r['best_val_macro_f1']:>12.4f} "
            f"{r['vindr_macro_f1']:>15.4f} "
            f"{r['inbreast_macro_f1']:>18.4f}"
        )

    return all_results


# =============================================================================
# Optional: class weights example
# Replace with your actual class distribution if needed
# Example:
#   class_weights = [1.0, 1.5, 4.0, 4.0, 6.0]
# =============================================================================

class_weights = None

results = run_experiments(
    train_loader=tr_dl,
    val_loader=val_dl,
    test_loader=ts_dl,
    inbreast_loader=inbreast_dl,
    class_weights=class_weights,
)

In [ ]:
ssssssss

In [ ]:

from tqdm import tqdm
import torch.nn as nn
from torchvision import models
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torch.optim import Adam
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
fffffff

In [ ]:
print("h1")